# Yelp — XGBoost Optimised
**Features**: 6 numerical + optimised ANN embeddings (from `optimization_ANN/`)
**Saves to**: `optimization_XGB/`


## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Check embedding files
STEP 3 — Load data
STEP 4 — Load optimised embeddings
STEP 5 — Build feature matrix
STEP 6 — Optimise & train
STEP 7 — Evaluate
STEP 8 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import optuna
from optuna.pruners  import MedianPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\data"
OPT_EMB  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\optimization_ANN"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\optimization_XGB"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Out dir: {OUT_DIR}")


---
## Step 2 — Check Embedding Files

In [ ]:
required = ['state_embeddings.npy','city_embeddings.npy','cat_embeddings.npy',
            'user_embeddings.npy','business_embeddings.npy',
            'state_encoder.json','city_encoder.json','cat_encoder.json',
            'user_encoder.json','business_encoder.json']
for fname in required:
    fpath  = os.path.join(OPT_EMB, fname)
    status = '✓' if os.path.exists(fpath) else '✗ MISSING — run ANN optimised first'
    print(f"  {status}  {fname}")


---
## Step 3 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


In [ ]:
NUM_COLS  = [
    'user_avg_rating',    'user_rating_count',
    'business_avg_rating','business_rating_count', 'business_rating_std',
    'days_since_review'
]
TARGET = 'stars'


---
## Step 4 — Load Optimised Embeddings

In [ ]:
OPT_EMB = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\optimization_ANN"
state_w = np.load(os.path.join(OPT_EMB,'state_embeddings.npy'))
city_w  = np.load(os.path.join(OPT_EMB,'city_embeddings.npy'))
cat_w   = np.load(os.path.join(OPT_EMB,'cat_embeddings.npy'))
user_w  = np.load(os.path.join(OPT_EMB,'user_embeddings.npy'))
biz_w   = np.load(os.path.join(OPT_EMB,'business_embeddings.npy'))
with open(os.path.join(OPT_EMB,'state_encoder.json'))    as f: state2idx = json.load(f)
with open(os.path.join(OPT_EMB,'city_encoder.json'))     as f: city2idx  = json.load(f)
with open(os.path.join(OPT_EMB,'cat_encoder.json'))      as f: cat2idx   = json.load(f)
with open(os.path.join(OPT_EMB,'user_encoder.json'))     as f: user2idx  = json.load(f)
with open(os.path.join(OPT_EMB,'business_encoder.json')) as f: biz2idx   = json.load(f)
print(f"State:{state_w.shape} City:{city_w.shape} Cat:{cat_w.shape} "
      f"User:{user_w.shape} Biz:{biz_w.shape}")


---
## Step 5 — Build Feature Matrix

In [ ]:
def lookup(series, encoder, weights):
    idx = series.map(lambda x: encoder.get(str(x), 0)).values
    return weights[idx]

def mean_pool_cats(series, cat2idx, cat_w):
    result = []
    for c_str in series.fillna('<UNK>'):
        tokens  = c_str.split('|')
        indices = [cat2idx.get(t, 0) for t in tokens]
        vecs    = cat_w[indices]
        result.append(vecs.mean(axis=0))
    return np.array(result, dtype=np.float32)

def build_matrix(df_, num_arr):
    return np.concatenate([
        num_arr,
        lookup(df_['state'],       state2idx, state_w),
        lookup(df_['city'],        city2idx,  city_w),
        mean_pool_cats(df_['categories'], cat2idx, cat_w),
        lookup(df_['user_id'],     user2idx,  user_w),
        lookup(df_['business_id'], biz2idx,   biz_w),
    ], axis=1)

scaler      = StandardScaler()
X_train_num = scaler.fit_transform(df_train[NUM_COLS])
X_val_num   = scaler.transform(df_val[NUM_COLS])
X_test_num  = scaler.transform(df_test[NUM_COLS])

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values

X_train = build_matrix(df_train, X_train_num)
X_val   = build_matrix(df_val,   X_val_num)
X_test  = build_matrix(df_test,  X_test_num)
print(f"Feature matrix — train: {X_train.shape}")


---
## Step 6 — Optimise & Train

In [ ]:
def get_metrics(y_true, y_pred):
    rmse = round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4)
    mae  = round(float(mean_absolute_error(y_true, y_pred)), 4)
    return rmse, mae


In [ ]:
SEED=42; MAX_ROUNDS=1000; EARLY_STOP=50

def objective(trial):
    params = {
        'max_depth'            : trial.suggest_int('max_depth',3,10),
        'learning_rate'        : trial.suggest_float('learning_rate',1e-3,0.3,log=True),
        'subsample'            : trial.suggest_float('subsample',0.5,1.0),
        'colsample_bytree'     : trial.suggest_float('colsample_bytree',0.5,1.0),
        'min_child_weight'     : trial.suggest_int('min_child_weight',1,10),
        'gamma'                : trial.suggest_float('gamma',0.0,5.0),
        'reg_alpha'            : trial.suggest_float('reg_alpha',1e-8,1.0,log=True),
        'reg_lambda'           : trial.suggest_float('reg_lambda',1e-8,1.0,log=True),
        'n_estimators'         : MAX_ROUNDS,
        'tree_method'          : 'hist',
        'random_state'         : SEED,
        'verbosity'            : 0,
        'early_stopping_rounds': EARLY_STOP,
    }
    m = XGBRegressor(**params)
    m.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=False)
    trial.set_user_attr('best_n_estimators',m.best_iteration+1)
    return float(np.sqrt(mean_squared_error(y_val,m.predict(X_val))))

study = optuna.create_study(direction='minimize',
    sampler=TPESampler(seed=SEED),pruner=MedianPruner(n_warmup_steps=10))
print("Starting Optuna — 50 trials...")
t_hpo = time.perf_counter()
study.optimize(objective,n_trials=50,show_progress_bar=True)
print(f"HPO done in {(time.perf_counter()-t_hpo)/60:.1f} min  Best: {study.best_value:.4f}")
best_n_est = study.best_trial.user_attrs['best_n_estimators']
p = {k:v for k,v in study.best_params.items()
     if k not in ['n_estimators','early_stopping_rounds']}
t0=time.perf_counter()
model=XGBRegressor(**p,n_estimators=best_n_est,tree_method='hist',random_state=SEED,verbosity=0)
model.fit(X_train,y_train)
train_time=time.perf_counter()-t0

---
## Step 7 — Evaluate

In [ ]:
tr_rmse,  tr_mae  = get_metrics(y_train, model.predict(X_train))
val_rmse, val_mae = get_metrics(y_val,   model.predict(X_val))
te_rmse,  te_mae  = get_metrics(y_test,  model.predict(X_test))

print("="*55)
print(f"XGBoost (Optimised) — Results")
print("="*55)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.2f}s")
print(f"  Train-Test gap : {round(te_rmse-tr_rmse,4)}")


---
## Step 8 — Save Results

In [ ]:
result = {
    'model'        : 'XGBoost (Optimised)',
    'train_rmse'   : tr_rmse, 'val_rmse'  : val_rmse, 'test_rmse'  : te_rmse,
    'train_mae'    : tr_mae,  'val_mae'   : val_mae,  'test_mae'   : te_mae,
    'train_time_s' : round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\optimization_XGB", 'xgb_results.csv'), index=False)
with open(os.path.join(r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\optimization_XGB", 'xgb_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Saved → C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\optimization_XGB")
print(f"Test RMSE: {te_rmse}  Test MAE: {te_mae}")
